In [ ]:
from config import client
import pandas as pd
from graph import plot_bar_graph, plot_line_graph, plot_pie_chart,plot_box_plot,plot_scatter_plot,plot_heatmap

In [ ]:
purchase_order_query = """
SELECT * FROM zoho_books_analytics.purchase_orders
"""

result = client.query(purchase_order_query)

purchase_order_df = pd.DataFrame(result.result_rows, columns=[col for col in result.column_names])

In [ ]:
purchase_order_df.columns

In [ ]:
columns_to_drop = ['purchase_order_id', 'vendor_id', 'attention_content',
        'reference_number', 'address_id',
       'currency_code', 'exchange_rate', 'delivery_instructions',
       'terms__conditions', 'date', 'sales_order_id',
       'crm_reference_id', 'source', 'total_fcy', 'sub_total_fcy',
        'customer_lot_code',
       'shipment_terms', 'delivery_terms', 'modified_by',
        'fpofin', 'po_re_confirmed', 
       'payment_terms', 'pricelist_id']
purchase_order_df.drop(columns=columns_to_drop, inplace=True)

In [ ]:
purchase_order_df['tariff_type'].value_counts()

In [ ]:
plot_bar_graph(purchase_order_df['purchase_order_status'], title="Purchase Order Status", x_label="Purchase Order Status", y_label="Count")

In [ ]:
purchase_order_df['shipment_preference'] = purchase_order_df['shipment_preference'].replace('', 'No Shipment Preference')
plot_bar_graph(purchase_order_df['shipment_preference'], title="Shipment Preference", x_label="Shipment Preference", y_label="Count")

In [ ]:
purchase_order_df['final_destination'] = purchase_order_df['final_destination'].replace('', 'No Final Destination')
plot_bar_graph(purchase_order_df['final_destination'], title="Final Destination", x_label="Final Destination", y_label="Count")

In [ ]:
purchase_order_df['billed_status'] = purchase_order_df['billed_status'].replace('', 'No Billed Status')
plot_bar_graph(purchase_order_df['billed_status'], title="Billed Status", x_label="Billed Status", y_label="Count")

In [ ]:
purchase_order_df['port_of_discharge'] = purchase_order_df['port_of_discharge'].replace('', 'No Port of Discharge')
plot_bar_graph(purchase_order_df['port_of_discharge'], title="Port of Discharge", x_label="Port of Discharge", y_label="Count")


In [ ]:
purchase_order_df['po_commited'] = purchase_order_df['po_commited'].replace('', 'No PO Committed')
plot_bar_graph(purchase_order_df['po_commited'], title="PO Committed", x_label="PO Committed", y_label="Count")

In [ ]:
purchase_order_df['tariff_'] = purchase_order_df['tariff_'].replace('', 'No Tariff')
plot_bar_graph(purchase_order_df['tariff_'], title="Tariff Type", x_label="Tariff Type", y_label="Count")

In [ ]:
purchase_order_df['tariff_type'] = purchase_order_df['tariff_type'].replace('', 'No Tariff')
plot_bar_graph(purchase_order_df['tariff_type'], title="Tariff Type", x_label="Tariff Type", y_label="Count")

In [ ]:
# Ensure datetime conversions
purchase_order_df['purchase_order_date'] = pd.to_datetime(purchase_order_df['purchase_order_date'], errors='coerce')
purchase_order_df['delivery_date'] = pd.to_datetime(purchase_order_df['delivery_date'], errors='coerce')
purchase_order_df['expected_delivery_date'] = pd.to_datetime(purchase_order_df['expected_delivery_date'], errors='coerce')
purchase_order_df['eta'] = pd.to_datetime(purchase_order_df['eta'], errors='coerce')
purchase_order_df['projected_etd'] = pd.to_datetime(purchase_order_df['projected_etd'], errors='coerce')
purchase_order_df['po_commited'] = pd.to_datetime(purchase_order_df['po_commited'], errors='coerce')

In [ ]:
# --------------------------------------------------------
# 1. PO Volume Over Time
po_volume = purchase_order_df.groupby(purchase_order_df['purchase_order_date'].dt.to_period('M'))['purchase_order_number'].count()
plot_line_graph(
    data_x=po_volume.index.astype(str),
    data_y=po_volume.values,
    title="PO Volume Over Time",
    x_label="Month",
    y_label="Number of POs"
)

In [ ]:
# --------------------------------------------------------
# 2. Expected vs Actual Delivery Date
plot_scatter_plot(
    x_data=purchase_order_df['expected_delivery_date'],
    y_data=purchase_order_df['delivery_date'],
    title="Expected vs Actual Delivery Date",
    x_label="Expected Delivery Date",
    y_label="Actual Delivery Date"
)

In [ ]:
# --------------------------------------------------------
# 3. ETA vs Projected ETD
plot_scatter_plot(
    x_data=purchase_order_df['projected_etd'],
    y_data=purchase_order_df['eta'],
    title="ETA vs Projected ETD",
    x_label="Projected ETD",
    y_label="ETA"
)

In [ ]:
# --------------------------------------------------------
# 4. On-Time vs Delayed Deliveries
purchase_order_df['delivery_delay_days'] = (purchase_order_df['delivery_date'] - purchase_order_df['expected_delivery_date']).dt.days
purchase_order_df['delivery_status'] = purchase_order_df['delivery_delay_days'].apply(lambda x: 'On Time' if x <= 0 else 'Delayed')

plot_bar_graph(
    purchase_order_df['delivery_status'],
    title="On-Time vs Delayed Deliveries",
    x_label="Delivery Status",
    y_label="Count"
)


In [ ]:
# --------------------------------------------------------
# 5. PO Commit Lead Time
purchase_order_df['po_commit_lead_days'] = (purchase_order_df['po_commited'] - purchase_order_df['purchase_order_date']).dt.days
plot_bar_graph(
    purchase_order_df['po_commit_lead_days'],
    title="PO Commit Lead Time Distribution",
    x_label="Lead Time (Days)",
    y_label="Count"
)

In [ ]:

# plot_box_plot(
#     data=purchase_order_df[['total_bcy']].join(purchase_order_df['tariff_type']),
#     title="PO Value by Tariff Type",
#     y_label="Total BCY"
# )
